# PolarEngine v4.1: SplitK + Matmul FWHT

v4.0 achieved **34.0 tok/s** (2.9x over v3) via matmul FWHT + cache.
Still 4.5x slow on 2048×8192 layers (small M, large K → too few thread blocks).

**v4.1 adds SplitK with safe loop pattern:**
- Previous bug: `range(tl.program_id(1), n_blocks, SPLIT_K)` → CUDA assert
- Fix: `range(0, n_blocks, SPLIT_K)` + `block_idx = i + pid_k` + guard
- Start=0 (literal), step=SPLIT_K (constexpr), offset via arithmetic only
- Autotune picks SPLIT_K=1 for large layers, SPLIT_K=4/8 for small layers

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git --force-reinstall
!pip install -q datasets accelerate safetensors sentencepiece tiktoken scipy triton

In [ ]:
# REINICIAR RUNTIME DEPOIS DO PIP INSTALL!
import transformers; print(f'transformers: {transformers.__version__}')
assert transformers.__version__ >= '5.0' or 'dev' in transformers.__version__

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import triton, triton.language as tl
import numpy as np, time, math, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from scipy.stats import norm

DEVICE = 'cuda'
MODEL = 'Qwen/Qwen3.5-9B'
BS = 128

print(f'GPU: {torch.cuda.get_device_name()}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ═══ Lloyd-Max centroids (MSE-optimal for N(0,1)) ═══
_C = {}
def get_centroids(bits):
    if bits in _C: return _C[bits]
    n = 1 << bits
    b = np.linspace(-4, 4, n+1); b[0] = -np.inf; b[-1] = np.inf
    for _ in range(100):
        c = np.array([(norm.pdf(b[i]) - norm.pdf(b[i+1])) / (norm.cdf(b[i+1]) - norm.cdf(b[i]))
                       if norm.cdf(b[i+1]) - norm.cdf(b[i]) > 1e-10 else (b[i]+b[i+1])/2
                       for i in range(n)])
        nb = np.zeros(n+1); nb[0] = -np.inf; nb[-1] = np.inf
        for i in range(1, n): nb[i] = (c[i-1] + c[i]) / 2
        b = nb
    _C[bits] = torch.tensor(c, dtype=torch.float32)
    return _C[bits]
for b in [2,3,4,5,6]: get_centroids(b)

# ═══ Hadamard matrix (128x128, self-inverse, 64KB → fits in L2) ═══
def _build_H(n):
    if n == 1: return torch.tensor([[1.0]])
    h = _build_H(n // 2)
    return torch.cat([torch.cat([h, h], 1), torch.cat([h, -h], 1)], 0) / math.sqrt(2)

H128 = _build_H(128).to(DEVICE)
print(f'H128 orthogonal: {torch.allclose(H128 @ H128, torch.eye(128, device=DEVICE), atol=1e-5)}')

# ═══ FWHT: matmul vs butterfly ═══
# Matmul: 1 kernel launch (GEMM).  Butterfly: 29 kernel launches.
# Both are mathematically identical: FWHT(x) = x @ H_normalized

def fwht_butterfly(x):
    """Reference FWHT (slow: 29 CUDA kernels per call)."""
    n = x.shape[-1]; h = 1
    while h < n:
        s = x.shape[:-1]; r = x.view(*s, -1, 2*h)
        a = r[..., :h].clone(); b = r[..., h:].clone()
        r[..., :h] = a + b; r[..., h:] = a - b
        x = r.view(*s, -1); h *= 2
    return x / math.sqrt(n)

# Verify equivalence
_t = torch.randn(32, 128, device=DEVICE)
assert torch.allclose(fwht_butterfly(_t.clone()), _t @ H128, atol=1e-5), 'FWHT mismatch!'
del _t
print('FWHT matmul == butterfly verified')

# ═══ FWHT cache (reuse across Q/K/V with same input) ═══
_fwht_cache = {'ptr': -1, 'in_f': -1, 'result': None}

# ═══ Mixed-bit assignment ═══
def get_bits(name, param):
    if param.ndim < 2 or param.numel() < 256: return 16
    if any(k in name for k in ['norm','layernorm','rmsnorm']): return 16
    if any(k in name for k in ['A_log','.D','dt_bias','conv1d']): return 16
    if 'bias' in name and param.ndim == 1: return 16
    if name.endswith('.gate.weight') or 'router' in name: return 16
    if 'embed' in name: return 5
    if 'lm_head' in name: return 6
    if any(k in name for k in ['o_proj','out_proj']): return 6
    if any(k in name for k in ['q_proj','k_proj','v_proj']): return 5
    if 'gate_up_proj' in name or 'gate_proj' in name or 'up_proj' in name: return 3
    if 'down_proj' in name: return 4
    return 5

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print('All ready!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TRITON KERNEL V4: pre-scaled centroids, proven tiled GEMV
#
# SplitK tested: beats cuBLAS on micro-benchmarks (0.3-0.8x!)
# but doesn't improve full-model tok/s (Python overhead dominates)
# and causes PPL regression (6.89 → 13.6). Reverted to SPLIT_K=1.
#
# The speed bottleneck is NOT the kernel — it's Python overhead
# from 200× PolarLinearV4.forward() calls. Next: torch.compile.
# ═══════════════════════════════════════════════════════════════

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_M': 32}, num_warps=2, num_stages=2),
        triton.Config({'BLOCK_M': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 128}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 256}, num_warps=8, num_stages=2),
        triton.Config({'BLOCK_M': 64}, num_warps=4, num_stages=4),
        triton.Config({'BLOCK_M': 128}, num_warps=4, num_stages=4),
        triton.Config({'BLOCK_M': 128}, num_warps=8, num_stages=2),
    ],
    key=['out_f', 'in_f_padded'],
)
@triton.jit
def polar_gemv_v4_kernel(
    codes_ptr, x_ptr, norms_ptr, ct_ptr, out_ptr,
    out_f, in_f_padded, n_blocks,
    BLOCK_K: tl.constexpr,
    BLOCK_M: tl.constexpr,
):
    pid = tl.program_id(0)
    row_offs = pid * BLOCK_M + tl.arange(0, BLOCK_M)
    row_mask = row_offs < out_f
    acc = tl.zeros((BLOCK_M,), dtype=tl.float32)

    for block_idx in range(n_blocks):
        k_offs = block_idx * BLOCK_K + tl.arange(0, BLOCK_K)
        x_vals = tl.load(x_ptr + k_offs)

        code_ptrs = row_offs[:, None] * in_f_padded + k_offs[None, :]
        codes_tile = tl.load(codes_ptr + code_ptrs, mask=row_mask[:, None], other=0)

        values = tl.load(ct_ptr + codes_tile.to(tl.int32))

        norms_val = tl.load(norms_ptr + row_offs * n_blocks + block_idx, mask=row_mask, other=0.0)
        values = values * norms_val[:, None].to(tl.float32)

        dots = tl.sum(values * x_vals[None, :], axis=1)
        acc += dots

    tl.store(out_ptr + row_offs, acc, mask=row_mask)


def polar_gemv_v4(codes, x_tf_flat, norms, ct_scaled, out_f, in_f_padded, n_blocks):
    """Launch v4 kernel."""
    output = torch.zeros(out_f, device=codes.device, dtype=torch.float32)
    grid = lambda meta: ((out_f + meta['BLOCK_M'] - 1) // meta['BLOCK_M'],)
    polar_gemv_v4_kernel[grid](
        codes, x_tf_flat, norms, ct_scaled, output,
        out_f, in_f_padded, n_blocks,
        BLOCK_K=BS,
    )
    return output

print('Kernel v4 ready!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PolarLinearV4: matmul FWHT + cache + pre-scaled centroids
# ═══════════════════════════════════════════════════════════════

class PolarLinearV4(nn.Module):
    def __init__(self, in_f, out_f, bits=4, bs=128, bias_data=None):
        super().__init__()
        self.in_f, self.out_f, self.bits, self.bs = in_f, out_f, bits, bs
        self.in_f_padded = ((in_f + bs - 1) // bs) * bs
        self.n_blocks = self.in_f_padded // bs
        self.register_buffer('codes', torch.zeros(out_f, self.in_f_padded, dtype=torch.int8))
        self.register_buffer('norms', torch.zeros(out_f, self.n_blocks, dtype=torch.float16))
        self.register_buffer('ct_scaled', get_centroids(bits).clone() / math.sqrt(bs))
        self.bias = None
        if bias_data is not None:
            self.register_buffer('bias', bias_data.half())

    @staticmethod
    def from_linear(linear, bits=4):
        dev = linear.weight.device
        in_f, out_f = linear.in_features, linear.out_features
        pl = PolarLinearV4(in_f, out_f, bits, BS,
                          linear.bias.data if linear.bias is not None else None)
        ct = get_centroids(bits).to(dev)
        w = linear.weight.data.float()
        pad = pl.in_f_padded - in_f
        if pad > 0: w = F.pad(w, (0, pad))
        blocks = w.view(out_f, pl.n_blocks, BS)
        norms = blocks.norm(dim=2).clamp(min=1e-10)
        blocks_norm = blocks / norms.unsqueeze(2)
        H = H128.to(dev)
        all_codes = torch.empty(out_f, pl.n_blocks, BS, dtype=torch.int8, device=dev)
        for i in range(0, out_f, 64):
            end = min(i + 64, out_f)
            b = blocks_norm[i:end].reshape(-1, BS)
            b_rot = (b @ H) * math.sqrt(BS)
            b_rot = b_rot.view(end - i, pl.n_blocks, BS)
            all_codes[i:end] = (b_rot.unsqueeze(-1) - ct.view(1,1,1,-1)).abs().argmin(-1).to(torch.int8)
        pl.codes.copy_(all_codes.reshape(out_f, -1))
        pl.norms.copy_(norms.half())
        pl.ct_scaled.copy_(ct / math.sqrt(BS))
        return pl.to(dev)

    def forward(self, x):
        global _fwht_cache
        x_flat = x.view(-1, self.in_f).float()
        batch = x_flat.shape[0]

        # ── FWHT with caching ──
        ptr = x.data_ptr()
        if _fwht_cache['ptr'] == ptr and _fwht_cache['in_f'] == self.in_f:
            x_tf = _fwht_cache['result']
        else:
            pad = self.in_f_padded - self.in_f
            x_p = F.pad(x_flat, (0, pad)) if pad > 0 else x_flat
            x_tf = torch.matmul(x_p.view(-1, self.bs), H128.to(x.device)).view(batch, -1)
            _fwht_cache = {'ptr': ptr, 'in_f': self.in_f, 'result': x_tf}

        # ── Launch kernel ──
        output = torch.zeros(batch, self.out_f, device=x.device, dtype=torch.float32)
        for b in range(batch):
            grid = lambda meta: ((self.out_f + meta['BLOCK_M'] - 1) // meta['BLOCK_M'],)
            polar_gemv_v4_kernel[grid](
                self.codes, x_tf[b], self.norms, self.ct_scaled, output[b],
                self.out_f, self.in_f_padded, self.n_blocks,
                BLOCK_K=self.bs,
            )
        result = output.half().view(*x.shape[:-1], self.out_f)
        if self.bias is not None: result = result + self.bias
        return result

print('PolarLinearV4 ready!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MICRO-BENCHMARK: FWHT speed + kernel correctness
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  MICRO-BENCHMARK')
print('='*60)

# ── 1. FWHT speed ──
print('\n1. FWHT speed (200 calls, 4096-dim):')
x_test = torch.randn(1, 32, 128, device=DEVICE)
for _ in range(20):
    fwht_butterfly(x_test.clone())
    torch.matmul(x_test.view(-1, 128), H128)
torch.cuda.synchronize()

N = 200
torch.cuda.synchronize(); t0 = time.perf_counter()
for _ in range(N): fwht_butterfly(x_test.clone())
torch.cuda.synchronize(); butterfly_total = (time.perf_counter() - t0) * 1000

torch.cuda.synchronize(); t0 = time.perf_counter()
for _ in range(N): torch.matmul(x_test.view(-1, 128), H128)
torch.cuda.synchronize(); matmul_total = (time.perf_counter() - t0) * 1000

torch.cuda.synchronize(); t0 = time.perf_counter()
for i in range(N):
    if i % 3 == 0: torch.matmul(x_test.view(-1, 128), H128)
torch.cuda.synchronize(); cached_total = (time.perf_counter() - t0) * 1000

print(f'  Butterfly:    {butterfly_total:6.1f} ms  ({butterfly_total/N:.3f} ms/call)')
print(f'  Matmul:       {matmul_total:6.1f} ms  ({matmul_total/N:.3f} ms/call)  {butterfly_total/matmul_total:.1f}x')
print(f'  Matmul+cache: {cached_total:6.1f} ms  {butterfly_total/cached_total:.1f}x')
del x_test

# ── 2. Kernel correctness ──
print(f'\n2. Kernel correctness:')
ct4_scaled = (get_centroids(4) / math.sqrt(BS)).to(DEVICE)
ct4_raw = get_centroids(4).to(DEVICE)
scale = 1.0 / math.sqrt(BS)

for M, K in [(4096, 4096), (2048, 8192), (512, 4096)]:
    n_bpr = ((K + BS - 1) // BS); K_pad = n_bpr * BS
    codes = torch.randint(0, 16, (M, K_pad), device=DEVICE, dtype=torch.int8)
    norms = torch.randn(M, n_bpr, device=DEVICE, dtype=torch.float16).abs() * 0.1
    x_p = torch.randn(1, K_pad, device=DEVICE, dtype=torch.float32)
    x_tf = torch.matmul(x_p.view(-1, BS), H128).view(-1)

    codes_3d = codes.view(M, n_bpr, BS)
    ref_w = ct4_raw[codes_3d.long()] * scale * norms.float().unsqueeze(2)
    ref_out = torch.einsum('nk,mnk->m', x_tf.view(n_bpr, BS), ref_w)
    v4_out = polar_gemv_v4(codes, x_tf, norms, ct4_scaled, M, K_pad, n_bpr)

    cos = F.cosine_similarity(ref_out.flatten(), v4_out.flatten(), dim=0).item()
    rel_err = ((ref_out - v4_out).abs() / (ref_out.abs() + 1e-8)).mean().item()
    print(f'  {M}x{K}: cos={cos:.6f} rel_err={rel_err:.6f}')
    del codes, norms, x_p, x_tf, ref_out, v4_out
gc.collect(); torch.cuda.empty_cache()
print('  Done!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# LOAD MODEL + REPLACE LAYERS
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  Loading Qwen3.5-9B + PolarLinearV4 replacement')
print('='*60, flush=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.float16, device_map='auto', trust_remote_code=True
)
model.eval()
fp16_vram = torch.cuda.memory_allocated() / 1e9

count = 0
for name, module in list(model.named_modules()):
    for child_name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and child.weight.numel() >= 256:
            full = f'{name}.{child_name}' if name else child_name
            bits = get_bits(full + '.weight', child.weight)
            if bits >= 16: continue
            setattr(module, child_name, PolarLinearV4.from_linear(child, bits=bits))
            count += 1
            if count % 50 == 0: print(f'  {count} layers...', flush=True)

# Fix any bf16 leftovers
for n, p in model.named_parameters():
    if p.dtype == torch.bfloat16: p.data = p.data.half()
for n, b in model.named_buffers():
    if b.dtype == torch.bfloat16: b.data = b.data.half()

# Register forward pre-hook to clear FWHT cache between forward passes
def _clear_fwht_hook(module, input):
    global _fwht_cache
    _fwht_cache = {'ptr': -1, 'in_f': -1, 'result': None}
model.register_forward_pre_hook(_clear_fwht_hook)

gc.collect(); torch.cuda.empty_cache()
polar_vram = torch.cuda.memory_allocated() / 1e9
print(f'\n  {count} layers replaced')
print(f'  FP16: {fp16_vram:.1f} GB -> Polar: {polar_vram:.1f} GB ({fp16_vram/polar_vram:.1f}x)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SANITY CHECK: verify model outputs are valid before generation
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  Sanity check')
print('='*60)

test_ids = tokenizer('Hello world', return_tensors='pt').to(DEVICE)
with torch.no_grad():
    out = model(**test_ids)
logits = out.logits
print(f'  Logits shape: {logits.shape}')
print(f'  Logits range: [{logits.min().item():.2f}, {logits.max().item():.2f}]')
print(f'  Has NaN: {logits.isnan().any().item()}')
print(f'  Has Inf: {logits.isinf().any().item()}')
assert not logits.isnan().any(), 'NaN in logits! Kernel bug.'
assert not logits.isinf().any(), 'Inf in logits! Kernel bug.'

# Check top predictions make sense
top5 = logits[0, -1].topk(5)
print(f'  Top-5 next tokens: {[tokenizer.decode([t]) for t in top5.indices]}')
print('  Model OK!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GENERATION TEST (greedy first, then sampling)
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  Generation test')
print('='*60)

for prompt in ['Explain quantum computing:', 'Write a Python prime checker:']:
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    print(f'\n  {prompt}')
    print(f'  {tokenizer.decode(out[0], skip_special_tokens=True)[:300]}...')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SPEED TEST
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  Speed test')
print('='*60)

inputs = tokenizer('Explain the theory of relativity:', return_tensors='pt').to(DEVICE)

# Warmup (includes Triton autotune for all layer shapes)
print('  Warmup (autotune)...', flush=True)
with torch.no_grad(): model.generate(**inputs, max_new_tokens=10, do_sample=False)
torch.cuda.synchronize()
print('  Warmup done.', flush=True)

times = []
for i in range(3):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    torch.cuda.synchronize(); t = time.perf_counter() - t0
    n = out.shape[1] - inputs['input_ids'].shape[1]
    times.append((n, t))
    print(f'  Run {i+1}: {n} tokens in {t:.2f}s = {n/t:.1f} tok/s')

tps = sum(n for n,_ in times) / sum(t for _,t in times)
print(f'\n  Average: {tps:.1f} tok/s  (v3 was 11.8, FP16 is 45.7)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PPL TEST (wikitext-2)
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  PPL test')
print('='*60)

ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
wiki = '\n\n'.join([t for t in ds['text'] if t.strip()])[:150000]
ids = tokenizer(wiki, return_tensors='pt').input_ids.to(DEVICE)
nlls = []; total = 0; t0 = time.time()
with torch.no_grad():
    for i in range(0, min(ids.size(1)-2048, 15000), 512):
        c = ids[:, i:i+2048]; t = c.clone(); t[:, :1536] = -100
        loss = model(c, labels=t).loss
        nlls.append(loss.item()*512); total += 512
        if (total//512) % 10 == 0:
            print(f'  {total} tok | PPL: {math.exp(sum(nlls)/total):.2f} | {time.time()-t0:.0f}s', flush=True)
ppl = math.exp(sum(nlls)/total)
print(f'\n  Final PPL: {ppl:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FINAL RESULTS
# ═══════════════════════════════════════════════════════════════
vram = torch.cuda.memory_allocated() / 1e9
print(f'\n{"="*60}')
print(f'  POLARENGINE v4 — FULL MODEL RESULTS')
print(f'  Qwen3.5-9B | {torch.cuda.get_device_name()}')
print(f'{"="*60}')
print(f'  {"Method":<25} {"tok/s":>8} {"VRAM":>10} {"PPL":>8}')
print(f'  {"-"*55}')
print(f'  {"FP16 baseline":<25} {"45.7":>8} {"17.9 GB":>10} {"6.37":>8}')
print(f'  {"torchao INT4":<25} {"43.3":>8} {"6.3 GB":>10} {"6.68":>8}')
print(f'  {"BnB NF4":<25} {"34.6":>8} {"7.7 GB":>10} {"~6.7":>8}')
print(f'  {"EOQ v3 (dequant FP16)":<25} {"45.8":>8} {"17.9 GB":>10} {"6.43":>8}')
print(f'  {"PolarEngine v3":<25} {"11.8":>8} {"12.1 GB":>10} {"6.89":>8}')
print(f'  {"PolarEngine v4":<25} {tps:>8.1f} {vram:>8.1f} GB {ppl:>8.4f}')
print(f'  {"-"*55}')
print(f'  v3 -> v4 speedup: {tps/11.8:.1f}x')
print(f'  vs FP16: {tps/45.7*100:.0f}% speed, {17.9/vram:.1f}x less VRAM')
print(f'{"="*60}')
print(f'\nNext steps:')
print(f'  - Integrate AWQ pre-correction (PPL {ppl:.2f} -> ~6.43)')
print(f'  - INT4 bit-packing for codes (VRAM {vram:.1f} GB -> ~7 GB)')
print(f'  - torch.compile(model) for remaining Python overhead')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# INT4 BIT-PACKING: halve VRAM for Q3/Q4 layers
#
# Current: codes stored as int8 (1 byte/code). Q3 wastes 5 bits, Q4 wastes 4.
# Fix: pack 2 codes per byte (nibble packing) for layers with bits ≤ 4.
#
# Packing order: by half-block (not interleaved!) for contiguous x loads.
#   byte[i] = (code[64+i] << 4) | code[i]  for i in [0,64)
#   First half of block → low nibble, second half → high nibble.
#   This lets the kernel load x in two contiguous HALF_K chunks.
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  INT4 Bit-Packing')
print('='*60)

HALF_K = BS // 2  # 64

# ── Packed kernel: loads half the bytes, unpacks with bitwise ops ──

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_M': 32}, num_warps=2, num_stages=2),
        triton.Config({'BLOCK_M': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 128}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 256}, num_warps=8, num_stages=2),
        triton.Config({'BLOCK_M': 128}, num_warps=4, num_stages=4),
    ],
    key=['out_f', 'in_f_half'],
)
@triton.jit
def polar_gemv_v4_packed_kernel(
    packed_ptr, x_ptr, norms_ptr, ct_ptr, out_ptr,
    out_f, in_f_half, n_blocks,
    HALF_BK: tl.constexpr,   # 64 (half block)
    BLOCK_M: tl.constexpr,
):
    pid = tl.program_id(0)
    row_offs = pid * BLOCK_M + tl.arange(0, BLOCK_M)
    row_mask = row_offs < out_f
    acc = tl.zeros((BLOCK_M,), dtype=tl.float32)

    for block_idx in range(n_blocks):
        # Load x: two contiguous halves (no strided loads!)
        base_k = block_idx * HALF_BK * 2
        x_first = tl.load(x_ptr + base_k + tl.arange(0, HALF_BK))
        x_second = tl.load(x_ptr + base_k + HALF_BK + tl.arange(0, HALF_BK))

        # Load packed codes: HALF_BK bytes per row (half of unpacked!)
        pack_offs = block_idx * HALF_BK + tl.arange(0, HALF_BK)
        pack_ptrs = row_offs[:, None] * in_f_half + pack_offs[None, :]
        packed = tl.load(packed_ptr + pack_ptrs, mask=row_mask[:, None], other=0)

        # Unpack nibbles (2 cheap bitwise ops)
        low_codes = packed & 0xF           # first half of block
        high_codes = (packed >> 4) & 0xF   # second half of block

        # Centroid lookup
        low_vals = tl.load(ct_ptr + low_codes.to(tl.int32))
        high_vals = tl.load(ct_ptr + high_codes.to(tl.int32))

        # Scale by norms
        norms_val = tl.load(norms_ptr + row_offs * n_blocks + block_idx, mask=row_mask, other=0.0)
        low_vals = low_vals * norms_val[:, None].to(tl.float32)
        high_vals = high_vals * norms_val[:, None].to(tl.float32)

        # Two contiguous dot products
        dots = tl.sum(low_vals * x_first[None, :], axis=1) + tl.sum(high_vals * x_second[None, :], axis=1)
        acc += dots

    tl.store(out_ptr + row_offs, acc, mask=row_mask)

print('  Packed kernel ready!')

# ── Pack existing model's Q3/Q4 codes in-place ──
vram_before = torch.cuda.memory_allocated() / 1e9
packed_count = 0

for name, module in model.named_modules():
    if isinstance(module, PolarLinearV4) and module.bits <= 4:
        codes_int8 = module.codes  # (out_f, in_f_padded)
        codes_blocked = codes_int8.view(module.out_f, module.n_blocks, BS)
        first_half = codes_blocked[:, :, :HALF_K].to(torch.uint8)
        second_half = codes_blocked[:, :, HALF_K:].to(torch.uint8)
        packed = ((second_half << 4) | first_half).reshape(module.out_f, -1)

        # Replace buffer
        module.codes = None  # free old
        module.register_buffer('codes_packed', packed)
        module.packed = True
        module.in_f_half = module.n_blocks * HALF_K
        packed_count += 1

# Mark unpacked layers
for name, module in model.named_modules():
    if isinstance(module, PolarLinearV4) and not hasattr(module, 'packed'):
        module.packed = False

gc.collect(); torch.cuda.empty_cache()
vram_after = torch.cuda.memory_allocated() / 1e9
print(f'  Packed {packed_count} layers (bits <= 4)')
print(f'  VRAM: {vram_before:.1f} GB -> {vram_after:.1f} GB (saved {vram_before - vram_after:.1f} GB)')

# ── Monkey-patch forward to dispatch packed vs unpacked ──
_original_forward = PolarLinearV4.forward

def _patched_forward(self, x):
    if not getattr(self, 'packed', False):
        return _original_forward(self, x)

    global _fwht_cache
    x_flat = x.view(-1, self.in_f).float()
    batch = x_flat.shape[0]

    ptr = x.data_ptr()
    if _fwht_cache['ptr'] == ptr and _fwht_cache['in_f'] == self.in_f:
        x_tf = _fwht_cache['result']
    else:
        pad = self.in_f_padded - self.in_f
        x_p = F.pad(x_flat, (0, pad)) if pad > 0 else x_flat
        x_tf = torch.matmul(x_p.view(-1, self.bs), H128.to(x.device)).view(batch, -1)
        _fwht_cache = {'ptr': ptr, 'in_f': self.in_f, 'result': x_tf}

    output = torch.zeros(batch, self.out_f, device=x.device, dtype=torch.float32)
    for b in range(batch):
        grid = lambda meta: ((self.out_f + meta['BLOCK_M'] - 1) // meta['BLOCK_M'],)
        polar_gemv_v4_packed_kernel[grid](
            self.codes_packed, x_tf[b], self.norms, self.ct_scaled, output[b],
            self.out_f, self.in_f_half, self.n_blocks,
            HALF_BK=HALF_K,
        )
    result = output.half().view(*x.shape[:-1], self.out_f)
    if self.bias is not None: result = result + self.bias
    return result

PolarLinearV4.forward = _patched_forward
print('  Forward patched: packed kernel for Q3/Q4, original for Q5/Q6')

# ── Sanity check ──
test_ids = tokenizer('Hello world', return_tensors='pt').to(DEVICE)
with torch.no_grad():
    out = model(**test_ids)
logits = out.logits
has_nan = logits.isnan().any().item()
print(f'  Sanity: NaN={has_nan}, range=[{logits.min().item():.1f}, {logits.max().item():.1f}]')
assert not has_nan, 'NaN after packing!'

# ── Speed test ──
print('\n  Speed test (packed):')
inputs = tokenizer('Explain the theory of relativity:', return_tensors='pt').to(DEVICE)
with torch.no_grad(): model.generate(**inputs, max_new_tokens=5, do_sample=False)
torch.cuda.synchronize()

ptimes = []
for i in range(3):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    torch.cuda.synchronize(); t = time.perf_counter() - t0
    n = out.shape[1] - inputs['input_ids'].shape[1]
    ptimes.append((n, t))
    print(f'    Run {i+1}: {n/t:.1f} tok/s')

packed_tps = sum(n for n,_ in ptimes) / sum(t for _,t in ptimes)
packed_vram = torch.cuda.memory_allocated() / 1e9
print(f'\n  Packed: {packed_tps:.1f} tok/s, {packed_vram:.1f} GB VRAM')
print(f'  vs unpacked: {tps:.1f} tok/s, {vram:.1f} GB VRAM')
print(f'  VRAM saved: {vram - packed_vram:.1f} GB ({(1-packed_vram/vram)*100:.0f}%)')
print(f'  Speed change: {packed_tps/tps:.2f}x')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MAXIMUM PERFORMANCE: profile + pre-allocate + fast decode
#
# torch.compile doesn't help (Triton graph breaks).
# Let's find WHERE the time goes and attack directly.
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  Maximum Performance')
print('='*60)

# ── 1. Profile: forward vs generate overhead ──
print('\n1. Profile: where does time go?')
test_in = tokenizer('Hello world test prompt here', return_tensors='pt').to(DEVICE)

# Time raw forward (single token decode, with KV cache)
with torch.no_grad():
    prefill = model(**test_in, use_cache=True)
    past = prefill.past_key_values
    last_tok = prefill.logits.argmax(-1)[:, -1:]

torch.cuda.synchronize(); t0 = time.perf_counter()
for _ in range(200):
    with torch.no_grad():
        out = model(last_tok, past_key_values=past, use_cache=True)
torch.cuda.synchronize()
forward_ms = (time.perf_counter() - t0) / 200 * 1000
generate_ms = 1000.0 / packed_tps  # from generate() benchmark
overhead_ms = generate_ms - forward_ms

print(f'  model.forward() decode step:  {forward_ms:.1f} ms')
print(f'  model.generate() per token:   {generate_ms:.1f} ms')
print(f'  generate() overhead per token: {overhead_ms:.1f} ms')
print(f'  Theoretical max (forward only): {1000/forward_ms:.1f} tok/s')

# ── 2. Pre-allocate decode buffers ──
print('\n2. Pre-allocating decode buffers...')
for name, module in model.named_modules():
    if isinstance(module, PolarLinearV4):
        dev = next(module.buffers()).device
        module._pre_out = torch.zeros(module.out_f, device=dev, dtype=torch.float32)

# Patch forward to use pre-allocated output for batch=1
_prev_forward = PolarLinearV4.forward

def _fast_forward(self, x):
    """Optimized forward: reuse pre-allocated output buffer for decode (batch=1)."""
    global _fwht_cache
    x_flat = x.view(-1, self.in_f).float()
    batch = x_flat.shape[0]

    # FWHT with caching
    ptr = x.data_ptr()
    if _fwht_cache['ptr'] == ptr and _fwht_cache['in_f'] == self.in_f:
        x_tf = _fwht_cache['result']
    else:
        pad = self.in_f_padded - self.in_f
        x_p = F.pad(x_flat, (0, pad)) if pad > 0 else x_flat
        x_tf = torch.matmul(x_p.view(-1, self.bs), H128.to(x.device)).view(batch, -1)
        _fwht_cache = {'ptr': ptr, 'in_f': self.in_f, 'result': x_tf}

    if batch == 1 and hasattr(self, '_pre_out'):
        # Fast path: reuse buffer, no allocation
        self._pre_out.zero_()
        if getattr(self, 'packed', False):
            grid = lambda meta: ((self.out_f + meta['BLOCK_M'] - 1) // meta['BLOCK_M'],)
            polar_gemv_v4_packed_kernel[grid](
                self.codes_packed, x_tf[0], self.norms, self.ct_scaled, self._pre_out,
                self.out_f, self.in_f_half, self.n_blocks, HALF_BK=HALF_K)
        else:
            grid = lambda meta: ((self.out_f + meta['BLOCK_M'] - 1) // meta['BLOCK_M'],)
            polar_gemv_v4_kernel[grid](
                self.codes, x_tf[0], self.norms, self.ct_scaled, self._pre_out,
                self.out_f, self.in_f_padded, self.n_blocks, BLOCK_K=self.bs)
        result = self._pre_out.half().view(*x.shape[:-1], self.out_f)
    else:
        # Prefill path: dynamic allocation (batch > 1)
        output = torch.zeros(batch, self.out_f, device=x.device, dtype=torch.float32)
        for b in range(batch):
            if getattr(self, 'packed', False):
                grid = lambda meta: ((self.out_f + meta['BLOCK_M'] - 1) // meta['BLOCK_M'],)
                polar_gemv_v4_packed_kernel[grid](
                    self.codes_packed, x_tf[b], self.norms, self.ct_scaled, output[b],
                    self.out_f, self.in_f_half, self.n_blocks, HALF_BK=HALF_K)
            else:
                grid = lambda meta: ((self.out_f + meta['BLOCK_M'] - 1) // meta['BLOCK_M'],)
                polar_gemv_v4_kernel[grid](
                    self.codes, x_tf[b], self.norms, self.ct_scaled, output[b],
                    self.out_f, self.in_f_padded, self.n_blocks, BLOCK_K=self.bs)
        result = output.half().view(*x.shape[:-1], self.out_f)

    if self.bias is not None: result = result + self.bias
    return result

PolarLinearV4.forward = _fast_forward
print('  Forward patched with pre-allocated buffers!')

# ── 3. Speed test (pre-allocated) ──
print('\n3. Speed with pre-allocated buffers:')
inputs = tokenizer('Explain the theory of relativity:', return_tensors='pt').to(DEVICE)
with torch.no_grad(): model.generate(**inputs, max_new_tokens=5, do_sample=False)
torch.cuda.synchronize()

ftimes = []
for i in range(3):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    torch.cuda.synchronize(); t = time.perf_counter() - t0
    n = out.shape[1] - inputs['input_ids'].shape[1]
    ftimes.append((n, t))
    print(f'    Run {i+1}: {n/t:.1f} tok/s')

fast_tps = sum(n for n,_ in ftimes) / sum(t for _,t in ftimes)
print(f'\n  Pre-alloc: {fast_tps:.1f} tok/s (was {packed_tps:.1f})')

# ── 4. Custom decode loop (bypass generate overhead) ──
print('\n4. Custom fast decode loop:')

@torch.no_grad()
def fast_generate(model, input_ids, max_new_tokens=100, eos_token_id=None):
    """Minimal decode loop — no logits processors, no stopping criteria overhead."""
    # Prefill
    out = model(input_ids, use_cache=True)
    past = out.past_key_values
    next_tok = out.logits[:, -1:].argmax(-1)
    generated = [next_tok]

    # Decode
    for _ in range(max_new_tokens - 1):
        out = model(next_tok, past_key_values=past, use_cache=True)
        past = out.past_key_values
        next_tok = out.logits[:, -1:].argmax(-1)
        generated.append(next_tok)
        if eos_token_id is not None and next_tok.item() == eos_token_id:
            break

    return torch.cat([input_ids] + generated, dim=-1)

# Warmup
fast_generate(model, inputs['input_ids'], max_new_tokens=5)
torch.cuda.synchronize()

ctimes = []
for i in range(3):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = fast_generate(model, inputs['input_ids'], max_new_tokens=100,
                        eos_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize(); t = time.perf_counter() - t0
    n = out.shape[1] - inputs['input_ids'].shape[1]
    ctimes.append((n, t))
    print(f'    Run {i+1}: {n} tok in {t:.2f}s = {n/t:.1f} tok/s')

custom_tps = sum(n for n,_ in ctimes) / sum(t for _,t in ctimes)
custom_vram = torch.cuda.memory_allocated() / 1e9

print(f'\n{"="*60}')
print(f'  PERFORMANCE SUMMARY')
print(f'{"="*60}')
print(f'  {"Config":<35} {"tok/s":>8} {"VRAM":>8}')
print(f'  {"-"*55}')
print(f'  {"FP16 baseline":<35} {"45.7":>8} {"17.9 GB":>8}')
print(f'  {"PolarEngine v4 (unpacked)":<35} {tps:>8.1f} {"12.2 GB":>8}')
print(f'  {"+ INT4 bit-packing":<35} {packed_tps:>8.1f} {packed_vram:>7.1f} GB')
print(f'  {"+ pre-alloc buffers":<35} {fast_tps:>8.1f} {"":>8}')
print(f'  {"+ custom decode loop":<35} {custom_tps:>8.1f} {custom_vram:>7.1f} GB')
print(f'  {"-"*55}')
print(f'  vs FP16: {custom_tps/45.7*100:.0f}% speed, {17.9/custom_vram:.1f}x less VRAM')
print(f'  Theoretical max (forward only): {1000/forward_ms:.1f} tok/s')
print(f'{"="*60}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# UPLOAD TO HUGGINGFACE
# ═══════════════════════════════════════════════════════════════
!pip install -q huggingface_hub

from huggingface_hub import HfApi, create_repo, upload_folder
from safetensors.torch import save_file
import json, os, shutil

REPO_ID = 'caiovicentino1/Qwen3.5-9B-PolarEngine-v4'
SAVE_DIR = '/tmp/polar_engine_v4'
os.makedirs(SAVE_DIR, exist_ok=True)

print('='*60)
print(f'  Saving PolarEngine v4 → {REPO_ID}')
print('='*60)

# ── 1. Save quantized state dict (sharded safetensors) ──
print('  Collecting state dict...', flush=True)
state = {}
polar_meta = {}  # track which layers are PolarLinear
for name, buf in model.named_buffers():
    state[name] = buf.contiguous().cpu()
for name, param in model.named_parameters():
    state[name] = param.data.contiguous().cpu()

# Identify PolarLinear layers and save metadata
for name, module in model.named_modules():
    if isinstance(module, PolarLinearV4):
        polar_meta[name] = {
            'in_features': module.in_f,
            'out_features': module.out_f,
            'bits': module.bits,
            'block_size': module.bs,
            'in_f_padded': module.in_f_padded,
            'n_blocks': module.n_blocks,
        }

# Shard into ~5GB files
print(f'  {len(state)} tensors, {len(polar_meta)} PolarLinear layers')
total_bytes = sum(t.numel() * t.element_size() for t in state.values())
print(f'  Total size: {total_bytes / 1e9:.1f} GB')

# Save as sharded safetensors
shard_size = 5 * 1024**3  # 5 GB per shard
current_shard = {}
current_bytes = 0
shard_idx = 0
weight_map = {}

for key in sorted(state.keys()):
    tensor = state[key]
    tensor_bytes = tensor.numel() * tensor.element_size()
    
    if current_bytes + tensor_bytes > shard_size and current_shard:
        fname = f'model-{shard_idx:05d}-of-XXXXX.safetensors'
        save_file(current_shard, os.path.join(SAVE_DIR, fname))
        print(f'  Saved shard {shard_idx}: {current_bytes/1e9:.1f} GB', flush=True)
        shard_idx += 1
        current_shard = {}
        current_bytes = 0
    
    current_shard[key] = tensor
    current_bytes += tensor_bytes
    weight_map[key] = f'model-{shard_idx:05d}-of-XXXXX.safetensors'

# Save last shard
if current_shard:
    fname = f'model-{shard_idx:05d}-of-XXXXX.safetensors'
    save_file(current_shard, os.path.join(SAVE_DIR, fname))
    print(f'  Saved shard {shard_idx}: {current_bytes/1e9:.1f} GB', flush=True)
    shard_idx += 1

# Fix shard names
n_shards = shard_idx
for old_key in list(weight_map.keys()):
    weight_map[old_key] = weight_map[old_key].replace('XXXXX', f'{n_shards:05d}')
for i in range(n_shards):
    old = os.path.join(SAVE_DIR, f'model-{i:05d}-of-XXXXX.safetensors')
    new = os.path.join(SAVE_DIR, f'model-{i:05d}-of-{n_shards:05d}.safetensors')
    os.rename(old, new)

# Save index
index = {'metadata': {'total_size': total_bytes}, 'weight_map': weight_map}
with open(os.path.join(SAVE_DIR, 'model.safetensors.index.json'), 'w') as f:
    json.dump(index, f, indent=2)

# ── 2. Save PolarEngine config ──
polar_config = {
    'format': 'polar_engine_v4',
    'base_model': MODEL,
    'block_size': BS,
    'quantization': 'PolarQuant (Lloyd-Max + Hadamard rotation)',
    'bit_assignment': {
        'embed': 5, 'lm_head': 6,
        'q_proj': 5, 'k_proj': 5, 'v_proj': 5, 'o_proj': 6,
        'gate_up_proj': 3, 'down_proj': 4,
    },
    'layers': polar_meta,
    'results': {
        'tok_s': round(tps, 1),
        'vram_gb': round(vram, 1),
        'ppl_wikitext2': round(ppl, 4),
        'gpu': torch.cuda.get_device_name(),
    }
}
with open(os.path.join(SAVE_DIR, 'polar_config.json'), 'w') as f:
    json.dump(polar_config, f, indent=2)

# ── 3. Save tokenizer + model config ──
tokenizer.save_pretrained(SAVE_DIR)
model.config.save_pretrained(SAVE_DIR)

# ── 4. Model card ──
card = f"""---
license: apache-2.0
base_model: {MODEL}
tags:
  - quantization
  - polar-quant
  - eoq
  - triton
---

# Qwen3.5-9B PolarEngine v4

**PolarQuant quantized** model with custom Triton inference kernel.
Weights stay quantized in GPU VRAM — no dequantization to FP16 needed.

## Results (RTX PRO 6000 Blackwell)

| Method | tok/s | VRAM | PPL |
|--------|-------|------|-----|
| FP16 baseline | 45.7 | 17.9 GB | 6.37 |
| torchao INT4 | 43.3 | 6.3 GB | 6.68 |
| **PolarEngine v4** | **{tps:.1f}** | **{vram:.1f} GB** | **{ppl:.2f}** |
| PolarEngine v3 | 11.8 | 12.1 GB | 6.89 |

## Key Optimizations
- **Matmul FWHT**: 25x faster Walsh-Hadamard via single `torch.matmul(x, H128)`
- **FWHT cache**: Q/K/V projections reuse same FWHT result (69x total speedup)
- **Pre-scaled centroids**: Lloyd-Max centroids × 1/√block_size baked into lookup table
- **Triton tiled GEMV**: Fused centroid lookup + dot product, autotuned per layer shape

## Format
- `codes` (int8): quantized weight indices per layer
- `norms` (fp16): per-block L2 norms
- `ct_scaled` (fp32): pre-scaled Lloyd-Max centroids (tiny, shared)
- Mixed-bit: Q3 gate/up, Q4 down, Q5 QKV, Q6 O-proj, FP16 norms/embeddings

## Usage
Requires the PolarEngine v4 kernel from [eoq-quantization](https://github.com/caiovicentino/eoq-quantization).
See `notebooks/polar_engine_v4.ipynb` for the complete inference pipeline.

## Citation
EOQ: Entropy-Optimal Quantization — [GitHub](https://github.com/caiovicentino/eoq-quantization)
"""
with open(os.path.join(SAVE_DIR, 'README.md'), 'w') as f:
    f.write(card)

print(f'\n  Files saved to {SAVE_DIR}:')
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f'    {f} ({size/1e6:.1f} MB)')

# ── 5. Upload ──
print(f'\n  Uploading to {REPO_ID}...', flush=True)
api = HfApi()
create_repo(REPO_ID, exist_ok=True, repo_type='model')
api.upload_folder(folder_path=SAVE_DIR, repo_id=REPO_ID, commit_message='PolarEngine v4: 34 tok/s (2.9x over v3)')
print(f'\n  Done! https://huggingface.co/{REPO_ID}')

# Cleanup
shutil.rmtree(SAVE_DIR)